In [ ]:
import os
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

# Optional: show GPU details
!nvidia-smi -L

## 1) Clone repo + install requirements
This installs everything needed (including `pyyaml` for `data.yaml`).

In [ ]:
# Clone (or re-clone)
!rm -rf /kaggle/working/TheGrid
!git clone https://github.com/SAUNAK359/TheGrid.git /kaggle/working/TheGrid

%cd /kaggle/working/TheGrid

# Install deps
!pip -q install -r requirements.txt

## 2) Place your `data.yaml` (YOLO)
You must create a YOLO-style `data.yaml`.

### Where should `data.yaml` be placed?
Put it here (recommended):
- **`/kaggle/working/data.yaml`**

Then we will run training with `--data /kaggle/working/data.yaml`.

### Example
If your Kaggle dataset looks like:
- `/kaggle/input/train/images/...`
- `/kaggle/input/train/labels/...`
- `/kaggle/input/val/images/...`
- `/kaggle/input/val/labels/...`
then set `train:` and `val:` to the **images folders**.

In [ ]:
from pathlib import Path

# Edit these two paths to match your Kaggle dataset
TRAIN_IMAGES = '/kaggle/input/train/images'
VAL_IMAGES   = '/kaggle/input/val/images'

# Class names (order defines class_id 0..N-1). Update to your dataset.
NAMES = ['class0']

data_yaml = f'''
path: /
train: {TRAIN_IMAGES}
val: {VAL_IMAGES}
names:
{chr(10).join([f'  - {n}' for n in NAMES])}
'''

out = Path('/kaggle/working/data.yaml')
out.write_text(data_yaml)
print('Wrote:', out)
print(out.read_text())

## 3) Train (YOLO-style CLI)
This uses the repo CLI: `python -m hybriddetector train ...`

Outputs (by default):
- Checkpoints → `checkpoints/` (inside the repo folder)

Tip: If you get out-of-memory, reduce `--batch` or `--img`.

In [ ]:
%cd /kaggle/working/TheGrid

# Recommended defaults for Kaggle GPU
EPOCHS = 50
BATCH = 8
IMG = 640

!python -m hybriddetector train \
  --data /kaggle/working/data.yaml \
  --epochs {EPOCHS} \
  --batch {BATCH} \
  --img {IMG} \
  --device cuda

## 4) Save/export the trained model
Training saves:
- `checkpoints/best_model.pth`
- `checkpoints/latest_checkpoint.pth`

This cell copies the best model into `/kaggle/working/` and also creates a ZIP for easy download.

In [ ]:
from pathlib import Path
import shutil

repo = Path('/kaggle/working/TheGrid')
ckpt = repo / 'checkpoints' / 'best_model.pth'
assert ckpt.exists(), f'Missing checkpoint: {ckpt}'

# Copy into Kaggle working output
dst = Path('/kaggle/working/best_model.pth')
shutil.copy2(ckpt, dst)
print('Saved:', dst)

# Optional: zip checkpoints/results
zip_base = Path('/kaggle/working/thegrid_artifacts')
shutil.make_archive(str(zip_base), 'zip', root_dir=str(repo), base_dir='checkpoints')
print('Zipped:', str(zip_base) + '.zip')

## 5) Predict / test on images
Run inference on a folder of images (or a single image).
Predictions are saved as `.jpg` with boxes.

In [ ]:
%cd /kaggle/working/TheGrid

SOURCE = '/kaggle/input/val/images'  # change to your test folder or single image path

!python -m hybriddetector predict \
  --weights checkpoints/best_model.pth \
  --source {SOURCE} \
  --data /kaggle/working/data.yaml \
  --device cuda \
  --save-dir /kaggle/working/preds

print('Saved visuals in /kaggle/working/preds')